In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
import time
import pandas as pd
import sqlite3

In [ ]:
service = Service()
options = Options()
options.headless = True
driver = webdriver.Firefox(service=service, options=options)

# data = []

url = "https://nutricost.com/search?type=product&options%5Bunavailable_products%5D=last&options%5Bprefix%5D=last&q=powder&options%5Bprefix%5D=last"
base_url = "https://nutricost.com"


driver.get(url)
time.sleep(2)

# Keep clicking "Load More" until it's gone
while True:
    try:
        load_more = driver.find_element(By.CLASS_NAME, "loadMore")
        driver.execute_script("arguments[0].click();", load_more)
        time.sleep(2)
    except NoSuchElementException:
        # Button not found, break the loop
        break
    except ElementClickInterceptedException:
        # Wait and try again if click is intercepted
        time.sleep(1)

# Extract all product URLs
links = set()
items = driver.find_elements(By.CLASS_NAME, "grid-view-item__link")
for item in items:
    href = item.get_attribute("href")
    if href:
        links.add(href if href.startswith("http") else base_url + href)

print(f"Found {len(links)} product URLs.")
for link in links:
    print(link)

driver.quit()

In [ ]:
from selenium.common.exceptions import NoSuchElementException

service = Service()
options = Options()
options.headless = True
driver = webdriver.Firefox(service=service, options=options)

data = []

for i, link in enumerate(links):
    try:
        driver.get(link)
        try:
            # Click Accept or Decline (choose one)
            accept_btn = driver.find_element(By.ID, "shopify-pc__banner__btn-decline")
            accept_btn.click()
            time.sleep(1)
        except NoSuchElementException:
            pass  # Popup not present, continue
        try:
            close_btn = driver.find_element(By.ID, "closeIconContainer")
            if close_btn.get_attribute("aria-label") == "Dismiss this popup":
                close_btn.click()
                time.sleep(1)
        except NoSuchElementException:
            pass  # Popup not present, continue
        time.sleep(1)  # Longer delay for stability
        title_elem = driver.find_element(By.CLASS_NAME, "product-single__title")
        title = title_elem.text
    except Exception as e:
        title = f"Error: {e}"
    data.append({ "title": title, "url": link})

for item in data:
    print(item)

driver.quit()

In [ ]:
import re

input_file = "nutricost_products"
output_file = "nutricost_products.csv"

with open(input_file, "r") as fin, open(output_file, "a") as fout:
    # Write header only if file is empty
    if fout.tell() == 0:
        fout.write("title,url\n")
    for line in fin:
        # Skip error lines
        if line.startswith("'title': 'Error:"):
            continue
        # Extract title and url using regex
        match = re.match(r"\{'title': '([^']+)', 'url': '([^']+)'\}", line.strip())
        if match:
            title, url = match.groups()
            # Escape quotes in title
            title = title.replace('"', '""')
            fout.write(f'"{title}","{url}"\n')

This is where we iterate though each link for product details